In [1]:
import pandas as pd

# Carregando a base tratada da Atividade 8
df = pd.read_csv("dados_tratados.csv")

# Conferindo as dimensões da base
print("Dimensões da base:", df.shape)

# Visualizando as primeiras linhas
display(df.head())

Dimensões da base: (392692, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [2]:
# Convertendo a coluna InvoiceDate para o formato de data
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

# Ordenando os registros pela data da compra
df = df.sort_values("InvoiceDate")

# Definindo o ponto de corte em 80% do período
data_corte = df["InvoiceDate"].quantile(0.80)

print("Data de corte:", data_corte)

Data de corte: 2011-11-01 15:55:00


In [3]:
# Separando os dados anteriores e posteriores à data de corte
dados_passado = df[df["InvoiceDate"] <= data_corte].copy()
dados_futuro = df[df["InvoiceDate"] > data_corte].copy()

print("Registros no período passado:", len(dados_passado))
print("Registros no período futuro:", len(dados_futuro))

Registros no período passado: 314183
Registros no período futuro: 78509


In [4]:
# Criando um conjunto com os clientes que realizaram compras no período futuro
clientes_futuros = set(dados_futuro["CustomerID"].unique())

print("Quantidade de clientes que compraram no futuro:", len(clientes_futuros))

Quantidade de clientes que compraram no futuro: 1883


In [5]:
# Criando as características dos clientes usando somente os dados do passado
X = dados_passado.groupby("CustomerID").agg(
    QuantidadeTotal=("Quantity", "sum"),
    PrecoMedio=("UnitPrice", "mean"),
    QuantidadePedidos=("InvoiceNo", "nunique"),
    Pais=("Country", "first")
).reset_index()

# Removendo o identificador do cliente das características
X = X.drop(columns=["CustomerID"])

# Criando o alvo:
# 1 = cliente realizou uma nova compra no período futuro
# 0 = cliente não realizou uma nova compra no período futuro
y = dados_passado.groupby("CustomerID").apply(
    lambda grupo: 1 if grupo["CustomerID"].iloc[0] in clientes_futuros else 0
)

print("Dimensões do X:", X.shape)
print("Dimensões do y:", y.shape)

print("\nColunas do X:")
print(X.columns.tolist())

print("\nPrimeiras linhas do X:")
display(X.head())

print("\nPrimeiros valores do y:")
display(y.head())

Dimensões do X: (3985, 4)
Dimensões do y: (3985,)

Colunas do X:
['QuantidadeTotal', 'PrecoMedio', 'QuantidadePedidos', 'Pais']

Primeiras linhas do X:


/tmp/ipykernel_14866/205741894.py:15: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  y = dados_passado.groupby("CustomerID").apply(


,QuantidadeTotal,PrecoMedio,QuantidadePedidos,Pais
0,74215,1.040000,1,United Kingdom
1,2266,2.734912,6,Iceland
2,2341,5.764839,4,Finland
3,197,3.841176,1,Norway
4,409,18.108286,7,Norway



Primeiros valores do y:


,0
CustomerID,
12346.0,0
12347.0,1
12348.0,0
12350.0,0
12352.0,1


## Expectativa inicial

Antes do treinamento, espero que o modelo consiga identificar alguns padrões de comportamento dos clientes que indiquem maior probabilidade de uma nova compra. Como o conjunto possui clientes que compraram novamente e clientes que não compraram novamente, espero obter um resultado melhor que o baseline, mas sem esperar uma precisão muito alta, já que estamos utilizando um primeiro modelo simples.

In [6]:
from sklearn.model_selection import train_test_split

# Separando os dados em treino e teste
# 80% para treinamento e 20% para teste
# stratify mantém a proporção das classes do y
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Treino:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTeste:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

Treino:
X_train: (3188, 4)
y_train: (3188,)

Teste:
X_test: (797, 4)
y_test: (797,)


In [7]:
print("Proporção das classes no conjunto de treino:")
print(y_train.value_counts(normalize=True))

print("\nProporção das classes no conjunto de teste:")
print(y_test.value_counts(normalize=True))

Proporção das classes no conjunto de treino:
0    0.61606
1    0.38394
Name: proportion, dtype: float64

Proporção das classes no conjunto de teste:
0    0.61606
1    0.38394
Name: proportion, dtype: float64


In [8]:
from sklearn.metrics import accuracy_score

# Baseline: sempre prevê a classe mais frequente no conjunto de treino
classe_majoritaria = y_train.mode()[0]

# Criando as previsões do baseline
y_pred_baseline = [classe_majoritaria] * len(y_test)

# Calculando a acurácia do baseline
acuracia_baseline = accuracy_score(y_test, y_pred_baseline)

print("Classe majoritária:", classe_majoritaria)
print("Acurácia do baseline:", acuracia_baseline)

Classe majoritária: 0
Acurácia do baseline: 0.616060225846926


## Escolha do primeiro modelo

Foi escolhida a Árvore de Decisão como primeiro modelo de Machine Learning. O problema foi estruturado como uma classificação binária, em que o objetivo é prever se o cliente realizará uma nova compra. A escolha também considera que o conjunto possui poucas características e que a árvore permite interpretar de forma simples quais condições influenciam as previsões

In [9]:
# Transformando a variável categórica "Pais" em valores numéricos
X = pd.get_dummies(X, columns=["Pais"], dtype=int)

print("Novas dimensões do X:", X.shape)
display(X.head())

Novas dimensões do X: (3985, 40)


,QuantidadeTotal,PrecoMedio,QuantidadePedidos,Pais_Australia,Pais_Austria,Pais_Bahrain,Pais_Belgium,Pais_Brazil,Pais_Canada,Pais_Channel Islands,...,Pais_RSA,Pais_Saudi Arabia,Pais_Singapore,Pais_Spain,Pais_Sweden,Pais_Switzerland,Pais_USA,Pais_United Arab Emirates,Pais_United Kingdom,Pais_Unspecified
0,74215,1.040000,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
1,2266,2.734912,6,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2341,5.764839,4,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,197,3.841176,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,409,18.108286,7,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [10]:
# Refazendo a divisão entre treino e teste após a transformação
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Novas dimensões:")
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

Novas dimensões:
X_train: (3188, 40)
X_test: (797, 40)


In [11]:
from sklearn.tree import DecisionTreeClassifier

# Criando o modelo
# max_depth limita a profundidade da árvore para manter o primeiro modelo simples
modelo = DecisionTreeClassifier(
    random_state=42,
    max_depth=5
)

# Treinando o modelo com os dados de treino
modelo.fit(X_train, y_train)

print("Modelo treinado com sucesso!")

Modelo treinado com sucesso!


In [12]:
# Realizando previsões com os dados de teste
y_pred = modelo.predict(X_test)

print("Previsões realizadas com sucesso!")
print("Primeiras 10 previsões:")
print(y_pred[:10])

Previsões realizadas com sucesso!
Primeiras 10 previsões:
[1 0 0 0 0 0 0 1 1 0]


In [13]:
# Criando uma tabela para comparar os valores reais com as previsões
comparacao = pd.DataFrame({
    "Valor real": y_test.values,
    "Previsão": y_pred
})

display(comparacao.head(10))

,Valor real,Previsão
0,1,1
1,0,0
2,0,0
3,1,0
4,0,0
5,1,0
6,1,0
7,1,1
8,1,1
9,0,0


In [14]:
from sklearn.metrics import accuracy_score

# Calculando a acurácia da Árvore de Decisão
acuracia_modelo = accuracy_score(y_test, y_pred)

print("Acurácia do modelo:", acuracia_modelo)
print("Acurácia do baseline:", acuracia_baseline)

Acurácia do modelo: 0.7314930991217063
Acurácia do baseline: 0.616060225846926


## Comparação dos resultados

O baseline apresentou uma acurácia de 61,61%, enquanto a Árvore de Decisão apresentou uma acurácia de 73,15% no conjunto de teste. Portanto, o primeiro modelo apresentou um resultado superior ao baseline, com uma diferença de 11,54 pontos percentuais.

Esse resultado ficou de acordo com a expectativa inicial de obter um desempenho melhor que o baseline. A diferença foi maior do que o esperado para um primeiro modelo simples, indicando que as características utilizadas conseguiram identificar alguns padrões relacionados à realização de novas compras.

## Dificuldades encontradas

Durante o desenvolvimento, ocorreu um erro ao tentar treinar a Árvore de Decisão porque a variável `Pais` continha valores em texto, como "Germany". O modelo não consegue trabalhar diretamente com esse tipo de dado. Para resolver, a variável categórica foi transformada em valores numéricos utilizando `get_dummies` e a divisão entre treino e teste foi refeita.

Também foi necessário ajustar a construção do conjunto de dados para evitar vazamento de informações, utilizando os dados anteriores à data de corte para criar as características e os dados posteriores para definir o resultado esperado.

In [15]:
# Resumo dos resultados do primeiro modelo

resultado = pd.DataFrame({
    "Modelo": ["Baseline", "Árvore de Decisão"],
    "Acurácia": [acuracia_baseline, acuracia_modelo]
})

display(resultado)

,Modelo,Acurácia
0,Baseline,0.616060
1,Árvore de Decisão,0.731493
